# w1 — EDA, Label Definition & Metrics Plan (W1)

**Owner:** Thái — Data & ML Evaluation · **Dataset:** PaySim (Kaggle) · **Task:** 5.4 / W1

Mục tiêu notebook:
1. Xác định nhãn gian lận chính xác (`isFraud`) và phân biệt với cờ rule có sẵn (`isFlaggedFraud`).
2. Phân tích class imbalance, loại giao dịch, amount và phân bố thời gian.
3. Chứng minh bằng số liệu vì sao accuracy không đủ → đề xuất PR-AUC, precision, recall, confusion matrix.
4. Audit leakage/availability cho từng cột → đầu vào cho `docs/feature-inventory.md`.

> Notebook dùng pandas trên một máy (≈1 GB RAM). Spark ETL thuộc phần của Đạt; số liệu ở đây dùng để đối chiếu với `01_data_profiling.ipynb`.

**Cách chạy:** đặt file CSV gốc ở `data/raw/` (không commit) hoặc khai báo biến môi trường `PAYSIM_CSV`, rồi chạy `jupyter nbconvert --to notebook --execute --inplace notebooks/02_eda_label_evaluation.ipynb`.

In [1]:
import os, json, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import average_precision_score

DATA_PATH = Path(os.environ.get("PAYSIM_CSV", "../data/raw/PS_20174392719_1491204439457_log.csv"))
FIG_DIR = Path("../reports/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = Path("../reports/eda_summary.json")
COMPUTE_CHECKSUM = True

# Giá trị kỳ vọng của file gốc (để cả nhóm đối chiếu khi tải lại)
EXPECTED = {"rows": 6_362_620, "bytes": 493_534_783,
            "sha256": "16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b"}

# Style biểu đồ: legit = xanh, fraud = cam (cố định trong mọi hình)
LEGIT, FRAUD = "#2a78d6", "#eb6834"
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "font.size": 9.5, "legend.frameon": False, "lines.linewidth": 2, "figure.dpi": 110,
})
pd.set_option("display.float_format", lambda x: f"{x:,.6g}")
pd.set_option("display.width", 140)

def compact(x, _=None):
    for div, suf in [(1e9, "B"), (1e6, "M"), (1e3, "K")]:
        if abs(x) >= div:
            return f"{x/div:.1f}".rstrip("0").rstrip(".") + suf
    return f"{x:,.0f}"

def save(fig, name):
    fig.savefig(FIG_DIR / name, dpi=160, bbox_inches="tight")

summary = {}
print("DATA_PATH =", DATA_PATH.resolve())

DATA_PATH = C:\Users\lenovo\AppData\Local\Temp\fraud-conda-bigdata-check-ej62a20h\source\data\raw\PS_20174392719_1491204439457_log.csv


## 1. Nạp dữ liệu và kiểm tra nguồn

In [2]:
size = DATA_PATH.stat().st_size
sha = None
if COMPUTE_CHECKSUM:
    h = hashlib.sha256()
    with open(DATA_PATH, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""):
            h.update(chunk)
    sha = h.hexdigest()

DTYPES = {"step": "int32", "type": "category", "amount": "float64", "nameOrig": "string",
          "oldbalanceOrg": "float64", "newbalanceOrig": "float64", "nameDest": "string",
          "oldbalanceDest": "float64", "newbalanceDest": "float64",
          "isFraud": "int8", "isFlaggedFraud": "int8"}
df = pd.read_csv(DATA_PATH, dtype=DTYPES)
# PaySim không có ID giao dịch: dùng số thứ tự dòng trong file gốc (0-based) làm transaction_id đề xuất
df["row_id"] = np.arange(len(df), dtype="int64")

src = {"file": DATA_PATH.name, "bytes": size, "sha256": sha, "rows": len(df), "columns": df.shape[1] - 1}
summary["source"] = src
print(json.dumps(src, indent=2))
print("Khớp file gốc:", size == EXPECTED["bytes"] and len(df) == EXPECTED["rows"]
      and (sha is None or sha == EXPECTED["sha256"]))

{
  "file": "PS_20174392719_1491204439457_log.csv",
  "bytes": 493534783,
  "sha256": "16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b",
  "rows": 6362620,
  "columns": 11
}
Khớp file gốc: True


In [3]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulls": df.isna().sum(),
    "distinct": df.nunique(),
    "example": df.iloc[2],
}).drop(index="row_id")
n_dup = int(df.drop(columns="row_id").duplicated().sum())
summary["quality"] = {"nulls_total": int(df.isna().sum().sum()), "duplicate_rows": n_dup}
print("Duplicate rows (toàn bộ cột):", n_dup)
schema

Duplicate rows (toàn bộ cột): 0


,dtype,nulls,distinct,example
step,int32,0,743,1
type,category,0,5,TRANSFER
amount,float64,0,5316900,181
nameOrig,string,0,6353307,C1305486145
oldbalanceOrg,float64,0,1845844,181
newbalanceOrig,float64,0,2682586,0
nameDest,string,0,2722362,C553264065
oldbalanceDest,float64,0,3614697,0
newbalanceDest,float64,0,3555499,0
isFraud,int8,0,2,1


## 2. Label definition: `isFraud` vs `isFlaggedFraud`

Theo mô tả dataset (Kaggle, PaySim):
- **`isFraud`** — giao dịch do agent gian lận thực hiện trong mô phỏng (chiếm quyền tài khoản khách, chuyển tiền sang tài khoản khác rồi rút tiền). Đây là **ground truth**.
- **`isFlaggedFraud`** — cờ của *rule nghiệp vụ* trong simulator: "chặn chuyển khoản > 200.000 trong một giao dịch". Đây là **output của một rule**, không phải nhãn.

Kiểm tra bằng dữ liệu:

In [4]:
ct = pd.crosstab(df["isFraud"], df["isFlaggedFraud"], margins=True)
display(ct)

flag = df["isFlaggedFraud"] == 1
rule_doc = (df["type"] == "TRANSFER") & (df["amount"] > 200_000)
flag_stats = {
    "flagged_rows": int(flag.sum()),
    "flagged_and_fraud": int((flag & (df.isFraud == 1)).sum()),
    "flag_types": df.loc[flag, "type"].astype(str).value_counts().to_dict(),
    "rows_matching_documented_rule_TRANSFER_gt_200k": int(rule_doc.sum()),
    "documented_rule_rows_flagged": int((rule_doc & flag).sum()),
    "flag_precision": float((flag & (df.isFraud == 1)).sum() / flag.sum()),
    "flag_recall": float((flag & (df.isFraud == 1)).sum() / (df.isFraud == 1).sum()),
}
summary["isFlaggedFraud"] = flag_stats
pd.Series(flag_stats)

isFlaggedFraud,0,1,All
isFraud,,,
0,6354407,0,6354407
1,8197,16,8213
All,6362604,16,6362620


flagged_rows                                                    16
flagged_and_fraud                                               16
flag_types                                        {'TRANSFER': 16}
rows_matching_documented_rule_TRANSFER_gt_200k              409110
documented_rule_rows_flagged                                    16
flag_precision                                                   1
flag_recall                                             0.00194813
dtype: object

In [5]:
# Giao dịch ngay sau mỗi TRANSFER bị flag: là CASH_OUT amount = 0 (bị chặn)
nxt = df.loc[df.index[flag.values] + 1, ["step", "type", "amount", "oldbalanceOrg", "isFraud", "isFlaggedFraud"]]
zero_amt = df[df["amount"] == 0]
summary["zero_amount_rows"] = {"rows": len(zero_amt), "fraud": int(zero_amt.isFraud.sum()),
                               "types": zero_amt["type"].astype(str).value_counts().to_dict(),
                               "all_follow_flagged_transfer": bool(set(zero_amt.index) == set(nxt.index))}
print(summary["zero_amount_rows"])
nxt.head()

{'rows': 16, 'fraud': 16, 'types': {'CASH_OUT': 16}, 'all_follow_flagged_transfer': True}


,step,type,amount,oldbalanceOrg,isFraud,isFlaggedFraud
2736447,212,CASH_OUT,0,0,1,0
3247298,250,CASH_OUT,0,0,1,0
3760289,279,CASH_OUT,0,0,1,0
5563714,387,CASH_OUT,0,0,1,0
5996408,425,CASH_OUT,0,0,1,0


**Kết luận nhãn**
- Nhãn huấn luyện/đánh giá: **`isFraud`** (1 = gian lận). Không dùng `isFlaggedFraud` làm nhãn.
- `isFlaggedFraud` chỉ bật ở **16** giao dịch (đều là TRANSFER, đều `isFraud = 1`) → precision 100% nhưng recall chỉ ~0,19%. Rule mô tả "TRANSFER > 200.000" khớp 409.110 dòng nhưng chỉ 16 dòng được flag → cờ **không tái tạo được** từ mô tả, nên chỉ dùng làm *baseline rule* để so sánh, **không làm feature** (nó chỉ được bật trên giao dịch gian lận → leakage).
- 16 dòng `amount = 0` đều là CASH_OUT gian lận nằm ngay sau một TRANSFER bị flag (giao dịch bị chặn) → ghi vào data-quality notes cho Đạt/Đức Anh.

## 3. Class imbalance

In [6]:
n = len(df); n_f = int(df.isFraud.sum())
imb = {"rows": n, "fraud": n_f, "legit": n - n_f, "fraud_rate": n_f / n,
       "legit_per_fraud": (n - n_f) / n_f,
       "accuracy_if_predict_all_legit": (n - n_f) / n}
summary["class_balance"] = imb
pd.Series(imb)

rows                            6.36262e+06
fraud                                 8,213
legit                           6.35441e+06
fraud_rate                       0.00129082
legit_per_fraud                     773.701
accuracy_if_predict_all_legit      0.998709
dtype: float64

Chỉ **0,129%** giao dịch là gian lận (≈ 1 : 774). Một "model" dự đoán *mọi giao dịch đều hợp lệ* đã đạt accuracy **99,87%** nhưng không bắt được giao dịch gian lận nào → accuracy không phản ánh mục tiêu cảnh báo (xem mục 8).

## 4. Loại giao dịch (`type`)

In [7]:
by_type = (df.groupby("type", observed=True)
             .agg(rows=("isFraud", "size"), fraud=("isFraud", "sum"),
                  fraud_rate=("isFraud", "mean"), amount_median=("amount", "median"))
             .sort_values("rows", ascending=False))
by_type["share_rows"] = by_type["rows"] / n
by_type["share_fraud"] = by_type["fraud"] / n_f
summary["by_type"] = by_type.reset_index().assign(type=lambda d: d["type"].astype(str)).to_dict("records")
by_type

,rows,fraud,fraud_rate,amount_median,share_rows,share_fraud
type,,,,,,
CASH_OUT,2237500,4116,0.00183955,"147,072",0.351663,0.501157
PAYMENT,2151495,0,0,"9,482.19",0.338146,0
CASH_IN,1399284,0,0,"143,428",0.219923,0
TRANSFER,532909,4097,0.00768799,"486,308",0.0837562,0.498843
DEBIT,41432,0,0,"3,048.99",0.00651178,0


In [8]:
order = by_type.index.astype(str).tolist()[::-1]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, col, color, title in [(axes[0], "rows", LEGIT, "Số giao dịch theo loại"),
                              (axes[1], "fraud", FRAUD, "Số giao dịch gian lận theo loại")]:
    vals = by_type.loc[order, col].values
    ax.barh(order, vals, height=0.5, color=color)
    ax.set_title(title)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(compact))
    ax.grid(axis="y", visible=False)
    for y, v in enumerate(vals):
        ax.text(v, y, "  " + (compact(v) if v else "0"), va="center", color=INK2, fontsize=9)
    ax.set_xlim(0, vals.max() * 1.18)
fig.text(0, -0.04, "Nguồn: PaySim, 6.362.620 giao dịch. Gian lận chỉ xuất hiện ở TRANSFER và CASH_OUT.",
         color=INK2, fontsize=8.5)
fig.tight_layout(); save(fig, "fig1_type.png"); plt.show()

C:\Users\lenovo\AppData\Local\Temp\ipykernel_30564\2488708293.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save(fig, "fig1_type.png"); plt.show()


- Gian lận **chỉ** có ở `TRANSFER` (4.097) và `CASH_OUT` (4.116); `CASH_IN`, `DEBIT`, `PAYMENT` có 0 ca.
- Trong tập con TRANSFER + CASH_OUT (2.770.409 dòng) tỷ lệ gian lận là 0,30%.
- Gợi ý cho Đạt (W3): có thể huấn luyện trên TRANSFER + CASH_OUT và gán score thấp/rule cho các loại còn lại — nhưng cần ghi rõ đây là quyết định dựa trên dữ liệu mô phỏng.

## 5. Amount

In [9]:
amt = df.groupby("isFraud")["amount"].describe(percentiles=[.01, .25, .5, .75, .99])
display(amt)
fr = df[df.isFraud == 1]
amount_notes = {"fraud_amount_max": float(fr.amount.max()),
                "fraud_rows_at_10M_cap": int((fr.amount == 1e7).sum()),
                "legit_amount_max": float(df.loc[df.isFraud == 0, "amount"].max()),
                "median_fraud": float(fr.amount.median()),
                "median_legit": float(df.loc[df.isFraud == 0, "amount"].median())}
summary["amount"] = amount_notes
amount_notes

,count,mean,std,min,1%,25%,50%,75%,99%,max
isFraud,,,,,,,,,,
0,6.35441e+06,"178,197","596,237",0.01,449.24,"13,368.4","74,684.7","208,365",1.58606e+06,9.24455e+07
1,"8,213",1.46797e+06,2.40425e+06,0,"1,842","127,091","441,423",1.51777e+06,1e+07,1e+07


{'fraud_amount_max': 10000000.0,
 'fraud_rows_at_10M_cap': 287,
 'legit_amount_max': 92445516.64,
 'median_fraud': 441423.44,
 'median_legit': 74684.72}

In [10]:
tc = df[df["type"].isin(["TRANSFER", "CASH_OUT"]) & (df["amount"] > 0)]
bins = np.linspace(0, 8, 65)
fig, ax = plt.subplots(figsize=(8, 3.4))
for lab, color, name in [(0, LEGIT, "Hợp lệ"), (1, FRAUD, "Gian lận")]:
    x = np.log10(tc.loc[tc.isFraud == lab, "amount"])
    ax.hist(x, bins=bins, density=True, histtype="step", linewidth=2, color=color, label=name)
ax.set_xticks(range(2, 9)); ax.set_xticklabels(["100", "1K", "10K", "100K", "1M", "10M", "100M"]); ax.set_xlim(1.8, 8)
ax.set_xlabel("amount (thang log)"); ax.set_ylabel("Mật độ")
ax.set_title("Phân bố amount — TRANSFER + CASH_OUT")
ax.legend(loc="upper left")
ax.annotate("Trần 10M: 287 ca gian lận", xy=(7.06, 0.29), xytext=(7.15, 0.55), color=INK2, fontsize=8.5,
            arrowprops=dict(arrowstyle="-", color=INK2, lw=0.8))
fig.tight_layout(); save(fig, "fig2_amount.png"); plt.show()

C:\Users\lenovo\AppData\Local\Temp\ipykernel_30564\2663174693.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save(fig, "fig2_amount.png"); plt.show()


- Giao dịch gian lận có amount lớn hơn rõ rệt (median ≈ 441K so với ≈ 75K của giao dịch hợp lệ).
- Amount gian lận bị **chặn trên ở 10.000.000** (287 ca đúng 10M) — đặc tính của simulator (fraud tách thành nhiều giao dịch ≤ 10M), không nên diễn giải như hành vi thật.
- Phân phối lệch phải mạnh → dùng `log(amount)` nếu dùng model tuyến tính (Logistic Regression).

## 6. Phân bố thời gian (`step` = 1 giờ mô phỏng)

In [11]:
per_step = df.groupby("step").agg(rows=("isFraud", "size"), fraud=("isFraud", "sum"))
per_step["legit"] = per_step["rows"] - per_step["fraud"]
df["hour"] = ((df["step"] - 1) % 24).astype("int8")      # giả định step 1 = 00:00 ngày 1
df["day"] = ((df["step"] - 1) // 24 + 1).astype("int16")
time_stats = {"step_min": int(df.step.min()), "step_max": int(df.step.max()),
              "distinct_steps": int(df.step.nunique()), "days": int(df.day.max()),
              "fraud_per_step_mean": float(per_step.fraud.mean()),
              "fraud_per_step_std": float(per_step.fraud.std()),
              "steps_with_only_fraud": int((per_step.legit == 0).sum()),
              "rows_sorted_by_step": bool(df.step.is_monotonic_increasing)}
cum = per_step["rows"].cumsum() / n
time_stats["step_at_70pct_rows"] = int((cum < 0.70).sum() + 1)
time_stats["step_at_85pct_rows"] = int((cum < 0.85).sum() + 1)
time_stats["step_at_95pct_rows"] = int((cum < 0.95).sum() + 1)
summary["time"] = time_stats
time_stats

{'step_min': 1,
 'step_max': 743,
 'distinct_steps': 743,
 'days': 31,
 'fraud_per_step_mean': 11.053835800807537,
 'fraud_per_step_std': 4.9986314256340725,
 'steps_with_only_fraud': 320,
 'rows_sorted_by_step': True,
 'step_at_70pct_rows': 323,
 'step_at_85pct_rows': 378,
 'step_at_95pct_rows': 490}

In [12]:
T1, T2 = time_stats["step_at_70pct_rows"], time_stats["step_at_85pct_rows"]
fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True)
axes[0].plot(per_step.index, per_step["legit"], color=LEGIT, linewidth=1.2)
axes[0].set_title("Giao dịch hợp lệ mỗi step (giờ)")
axes[1].plot(per_step.index, per_step["fraud"], color=FRAUD, linewidth=1.2)
axes[1].set_title("Giao dịch gian lận mỗi step (giờ)")
axes[1].set_xlabel("step")
for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(compact))
    for t in (T1, T2):
        ax.axvline(t + 0.5, color=INK2, linewidth=0.8)
for x, lab in [((1 + T1) / 2, "train"), ((T1 + T2) / 2, "val"), ((T2 + 743) / 2, "test")]:
    axes[0].text(x, axes[0].get_ylim()[1] * 0.92, lab, ha="center", color=INK2, fontsize=9)
fig.tight_layout(); save(fig, "fig3_time_step.png"); plt.show()

C:\Users\lenovo\AppData\Local\Temp\ipykernel_30564\679059247.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save(fig, "fig3_time_step.png"); plt.show()


In [13]:
by_hour = df.groupby("hour").agg(rows=("isFraud", "size"), fraud=("isFraud", "sum"), fraud_rate=("isFraud", "mean"))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].bar(by_hour.index, by_hour["rows"], width=0.6, color=LEGIT)
axes[0].set_title("Số giao dịch theo giờ trong ngày")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(compact))
axes[1].bar(by_hour.index, by_hour["fraud_rate"] * 100, width=0.6, color=FRAUD)
axes[1].set_title("Tỷ lệ gian lận theo giờ trong ngày (%)")
axes[1].set_ylim(0, by_hour["fraud_rate"].max() * 100 * 1.15)
for ax in axes:
    ax.set_xticks(range(0, 24, 3)); ax.set_xlabel("giờ (step mod 24)"); ax.grid(axis="x", visible=False)
peak = by_hour["fraud_rate"].idxmax()
axes[1].text(peak, by_hour.loc[peak, "fraud_rate"] * 100, f"{by_hour.loc[peak, 'fraud_rate']*100:.1f}%",
             ha="center", va="bottom", color=INK2, fontsize=8.5)
fig.tight_layout(); save(fig, "fig4_hour.png"); plt.show()
summary["by_hour_fraud_rate"] = {int(k): float(v) for k, v in by_hour["fraud_rate"].items()}
by_hour.T

C:\Users\lenovo\AppData\Local\Temp\ipykernel_30564\684482696.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save(fig, "fig4_hour.png"); plt.show()


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
rows,"27,111","9,018","2,007","1,241","1,641","3,420","8,988","26,915","283,518","425,729",...,"416,686","441,612","439,941","580,509","647,814","553,728","247,806","194,555","141,257","71,587"
fraud,358,372,326,274,366,358,328,368,341,375,...,341,345,353,343,342,340,347,351,323,300
fraud_rate,0.013205,0.0412508,0.162431,0.22079,0.223035,0.104678,0.0364931,0.0136727,0.00120275,0.000880842,...,0.000818362,0.000781229,0.00080238,0.000590861,0.000527929,0.00061402,0.00140029,0.00180412,0.00228661,0.00419071


- `step` chạy từ 1 đến 743 (≈ 31 ngày, mỗi step = 1 giờ). Dữ liệu **đã sắp xếp theo step**, không có timestamp chi tiết hơn giờ.
- Gian lận xuất hiện **gần như đều theo thời gian** (~11 ca/step), trong khi giao dịch hợp lệ dao động mạnh theo chu kỳ ngày và **gần như biến mất sau ~step 400**; 320 step chỉ có giao dịch gian lận.
- Hệ quả: tỷ lệ gian lận lúc 2–5 giờ sáng lên tới ~22%, và nửa sau của tháng có prevalence cao hơn nhiều. Đây là **artifact của simulator** → feature `hour` có thể "học vẹt" artifact; split theo thời gian sẽ có prevalence khác nhau giữa train/val/test (mục 9).
- Đề xuất event-time cho streaming: `event_time = T0 + (step − 1) giờ` (T0 do Lead chốt trong `decisions.md`); nếu cần thời gian trong giờ, phân bổ đều theo thứ tự dòng và ghi rõ là mô phỏng.

## 7. Tài khoản (`nameOrig`, `nameDest`)

In [14]:
orig_counts = df["nameOrig"].value_counts()
dest_counts = df["nameDest"].value_counts()
acct = {"nameOrig_distinct": int(df.nameOrig.nunique()),
        "nameOrig_repeated_accounts": int((orig_counts > 1).sum()),
        "nameDest_distinct": int(df.nameDest.nunique()),
        "nameDest_median_txn": float(dest_counts.median()), "nameDest_max_txn": int(dest_counts.max()),
        "orig_prefix": df.nameOrig.str[0].value_counts().to_dict(),
        "dest_prefix": df.nameDest.str[0].value_counts().to_dict(),
        "merchant_dest_only_in_PAYMENT": bool((df.loc[df.nameDest.str.startswith("M"), "type"] == "PAYMENT").all()),
        "merchant_dest_balance_always_0": bool((df.loc[df.nameDest.str.startswith("M"), ["oldbalanceDest", "newbalanceDest"]] == 0).all().all())}
tr_f = df[(df.isFraud == 1) & (df.type == "TRANSFER")]; co_f = df[(df.isFraud == 1) & (df.type == "CASH_OUT")]
acct["fraud_TRANSFER_dest_is_fraud_CASHOUT_orig"] = int(tr_f.nameDest.isin(co_f.nameOrig).sum())
acct["fraud_TRANSFER_with_CASHOUT_same_step_amount"] = int(
    tr_f.set_index(["step", "amount"]).index.isin(co_f.set_index(["step", "amount"]).index).sum())
summary["accounts"] = acct
acct

{'nameOrig_distinct': 6353307,
 'nameOrig_repeated_accounts': 9298,
 'nameDest_distinct': 2722362,
 'nameDest_median_txn': 1.0,
 'nameDest_max_txn': 113,
 'orig_prefix': {'C': 6362620},
 'dest_prefix': {'C': 4211125, 'M': 2151495},
 'merchant_dest_only_in_PAYMENT': True,
 'merchant_dest_balance_always_0': True,
 'fraud_TRANSFER_dest_is_fraud_CASHOUT_orig': 0,
 'fraud_TRANSFER_with_CASHOUT_same_step_amount': 4078}

- `nameOrig` gần như là duy nhất (6.353.307 giá trị / 6.362.620 dòng; chỉ 9.298 tài khoản xuất hiện > 1 lần) → **feature lịch sử theo tài khoản gửi gần như không khả dụng**.
- `nameDest` lặp lại nhiều hơn (2,72 triệu giá trị, tối đa 113 lần) → feature lịch sử theo tài khoản nhận có thể thử ở W5–W6 (chỉ dùng dữ liệu *trước* thời điểm dự đoán).
- Tài khoản `M…` (merchant) chỉ xuất hiện ở PAYMENT và luôn có số dư 0 (không có thông tin).
- Chuỗi gian lận TRANSFER → CASH_OUT **không nối được bằng ID** (0 trường hợp `nameDest` của TRANSFER = `nameOrig` của CASH_OUT), nhưng 4.078/4.097 TRANSFER gian lận có một CASH_OUT gian lận cùng step và cùng amount → cặp giao dịch là artifact của simulator, cần thận trọng nếu xây feature từ nó.

## 8. Leakage audit — các cột số dư

Mô tả dataset lưu ý: *giao dịch bị phát hiện gian lận sẽ bị hủy*, nên các cột số dư không đáng tin cho fraud detection. Kiểm tra trên TRANSFER + CASH_OUT:

In [15]:
tc = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
tc["drain_orig"] = (tc.amount == tc.oldbalanceOrg) & (tc.amount > 0)                 # rút sạch số dư (pre-transaction)
tc["dest_zero_both"] = (tc.oldbalanceDest == 0) & (tc.newbalanceDest == 0) & (tc.amount > 0)
tc["orig_balance_consistent"] = (tc.oldbalanceOrg - tc.amount - tc.newbalanceOrig).abs() <= 0.01  # dùng post-transaction
tc["dest_balance_consistent"] = (tc.oldbalanceDest + tc.amount - tc.newbalanceDest).abs() <= 0.01
signals = ["drain_orig", "dest_zero_both", "orig_balance_consistent", "dest_balance_consistent"]
leak = tc.groupby("isFraud")[signals].mean().T.rename(columns={0: "legit_share", 1: "fraud_share"})
summary["leakage_signals_TRANSFER_CASHOUT"] = leak.to_dict("index")
leak

isFraud,legit_share,fraud_share
drain_orig,0,0.976257
dest_zero_both,0.000617625,0.495556
orig_balance_consistent,0.0949209,0.994521
dest_balance_consistent,0.774239,0.435042


In [16]:
labels = {"drain_orig": "amount = oldbalanceOrg (rút sạch số dư)",
          "dest_zero_both": "oldbalanceDest = newbalanceDest = 0",
          "orig_balance_consistent": "newbalanceOrig = oldbalanceOrg − amount",
          "dest_balance_consistent": "newbalanceDest = oldbalanceDest + amount"}
y = np.arange(len(signals))[::-1]
fig, ax = plt.subplots(figsize=(9, 3.4))
h = 0.32
ax.barh(y + h / 2 + 0.02, leak["legit_share"] * 100, height=h, color=LEGIT, label="Hợp lệ")
ax.barh(y - h / 2 - 0.02, leak["fraud_share"] * 100, height=h, color=FRAUD, label="Gian lận")
for yy, s in zip(y, signals):
    for off, col in [(h / 2 + 0.02, "legit_share"), (-h / 2 - 0.02, "fraud_share")]:
        ax.text(leak.loc[s, col] * 100 + 1, yy + off, f"{leak.loc[s, col]*100:.1f}%", va="center", color=INK2, fontsize=8.5)
ax.set_yticks(y); ax.set_yticklabels([labels[s] for s in signals])
ax.set_xlim(0, 112); ax.set_xlabel("% giao dịch thỏa điều kiện (TRANSFER + CASH_OUT)")
ax.set_title("Tín hiệu từ cột số dư: tách lớp gần như hoàn hảo")
ax.grid(axis="y", visible=False); ax.legend(loc="lower right", bbox_to_anchor=(1, 1), ncol=2)
fig.tight_layout(); save(fig, "fig5_leakage.png"); plt.show()

C:\Users\lenovo\AppData\Local\Temp\ipykernel_30564\1580003412.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); save(fig, "fig5_leakage.png"); plt.show()


- `amount = oldbalanceOrg` đúng với **97,6%** giao dịch gian lận và **0** giao dịch hợp lệ trong TRANSFER/CASH_OUT → một phép so sánh bằng đã gần "giải" bài toán. Đây là dấu hiệu **artifact của simulator** (fraud luôn rút sạch tài khoản), giải thích vì sao nhiều notebook công khai báo cáo ~99% — không nên coi là hiệu năng thực tế.
- `newbalanceOrig`/`newbalanceDest` là giá trị **sau giao dịch** → **không có tại thời điểm chấm điểm**; ngoài ra cách simulator cập nhật số dư khác hẳn giữa hai lớp (99,5% giao dịch gian lận khớp công thức `oldbalanceOrg − amount`, chỉ 9,5% giao dịch hợp lệ khớp) ⇒ loại khỏi feature, kể cả các feature "balance error" suy ra từ chúng.
- `oldbalanceOrg`/`oldbalanceDest` là giá trị **trước giao dịch** → về nguyên tắc có tại thời điểm scoring, nhưng mang artifact mạnh ⇒ đề xuất dùng ở dạng *ablation* (feature set B) và luôn báo cáo kèm feature set A không có số dư.

## 9. Vì sao accuracy không đủ — so sánh các baseline "ngây thơ"

Tất cả các "model" dưới đây đều là rule, không huấn luyện; mục đích là cho thấy accuracy gần như không đổi trong khi precision/recall thay đổi hoàn toàn.

In [17]:
def evaluate(y_true, y_pred, score=None, amount=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum()); fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum()); tn = int(((y_pred == 0) & (y_true == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
    out = {"accuracy": (tp + tn) / len(y_true), "precision": prec, "recall": rec,
           "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
           "pr_auc": average_precision_score(y_true, y_pred if score is None else score),
           "alerts": tp + fp, "TP": tp, "FP": fp, "FN": fn, "TN": tn}
    if amount is not None:
        amount = np.asarray(amount)
        out["fraud_amount_recall"] = amount[(y_pred == 1) & (y_true == 1)].sum() / amount[y_true == 1].sum()
    return out

y = df["isFraud"].values
is_tc = df["type"].isin(["TRANSFER", "CASH_OUT"]).values
drain = ((df.amount == df.oldbalanceOrg) & (df.amount > 0) & is_tc).values
baselines = {
    "Dự đoán tất cả hợp lệ": np.zeros(n, dtype=int),
    "isFlaggedFraud (rule có sẵn)": df["isFlaggedFraud"].values,
    "type ∈ {TRANSFER, CASH_OUT}": is_tc.astype(int),
    "amount = oldbalanceOrg (artifact)": drain.astype(int),
}
res = pd.DataFrame({k: evaluate(y, v, amount=df.amount.values) for k, v in baselines.items()}).T
summary["naive_baselines"] = res.to_dict("index")
res[["accuracy", "precision", "recall", "f1", "pr_auc", "fraud_amount_recall", "alerts", "TP", "FP", "FN"]]

,accuracy,precision,recall,f1,pr_auc,fraud_amount_recall,alerts,TP,FP,FN
Dự đoán tất cả hợp lệ,0.998709,0,0,0,0.00129082,0,0,0,0,"8,213"
isFlaggedFraud (rule có sẵn),0.998712,1,0.00194813,0.00388869,0.00323644,0.0064518,16,16,0,"8,197"
"type ∈ {TRANSFER, CASH_OUT}",0.565871,0.00296454,1,0.00591156,0.00296454,1,2.77041e+06,"8,213",2.7622e+06,0
amount = oldbalanceOrg (artifact),0.999969,1,0.976257,0.987986,0.976288,0.877366,"8,018","8,018",0,195


- Accuracy của 3/4 baseline đều ≥ 99,7% nhưng recall đi từ **0%** đến **100%** và precision từ **0,3%** đến **100%** → accuracy không phân biệt được các hệ thống cảnh báo.
- PR-AUC của bộ phân loại ngẫu nhiên ≈ tỷ lệ gian lận (0,0013), nên PR-AUC phải luôn được đọc **so với prevalence** của đúng split.
- Rule `amount = oldbalanceOrg` cho kết quả gần hoàn hảo → xác nhận rủi ro "too good to be true" ở mục 8; nếu model W3 đạt PR-AUC ~0,99, cần chạy ablation không có cột số dư trước khi kết luận.

## 10. Đề xuất split theo thời gian và prevalence từng phần

In [18]:
cuts = [0, T1, T2, int(df.step.max())]
split = pd.cut(df["step"], cuts, labels=["train", "val", "test"])
sp = df.groupby(split, observed=True).agg(step_from=("step", "min"), step_to=("step", "max"),
                                          rows=("isFraud", "size"), fraud=("isFraud", "sum"),
                                          fraud_rate=("isFraud", "mean"))
sp["share_rows"] = sp["rows"] / n; sp["share_fraud"] = sp["fraud"] / n_f
summary["time_split_proposal"] = sp.reset_index().assign(step=lambda d: d["step"].astype(str)).to_dict("records")
sp

,step_from,step_to,rows,fraud,fraud_rate,share_rows,share_fraud
step,,,,,,,
train,1,323,4463587,3643,0.00081616,0.701533,0.443565
val,324,378,980416,564,0.000575266,0.15409,0.0686716
test,379,743,918617,4006,0.0043609,0.144377,0.487763


- Split theo thời gian ~70/15/15 theo số dòng: **train step 1–323, val 324–378, test 379–743**. Prevalence ở test cao hơn train khoảng 5 lần và test chứa gần nửa số ca gian lận — vì giao dịch hợp lệ thưa dần ở cuối tháng.
- Đề xuất: dùng split thời gian làm **protocol chính** (khớp với kịch bản streaming: học quá khứ, replay tương lai qua Kafka), chọn threshold trên val, chỉ đánh giá test một lần; báo cáo thêm một split stratified-random để so sánh độ ổn định. Chi tiết trong `docs/evaluation-plan.md`.

## 11. Lưu summary cho báo cáo

In [19]:
def to_py(o):
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return o
SUMMARY_PATH.write_text(json.dumps(to_py(summary), indent=2, ensure_ascii=False))
print("Đã ghi", SUMMARY_PATH, "và", sorted(p.name for p in FIG_DIR.glob("*.png")))

Đã ghi ..\reports\eda_summary.json và ['fig1_type.png', 'fig2_amount.png', 'fig3_time_step.png', 'fig4_hour.png', 'fig5_leakage.png']
